# Unlimited-OCR-2 — document OCR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Unlimited-OCR-2-document-ocr.ipynb)

[`baidu/Unlimited-OCR-2`](https://huggingface.co/baidu/Unlimited-OCR-2) is Baidu's **~3.34B** vision-language OCR model (MIT, not gated). It ships as custom code (`UnlimitedOCRForCausalLM`) and the published Transformers path is `AutoModel` + `AutoTokenizer` with `trust_remote_code`. The card loads **bfloat16** and was tested on an NVIDIA A800-80GB. A free T4 has no native bfloat16, so this notebook loads **float16** and stays on the published `infer()` method.

Why this model: Unlimited-OCR-2 is aimed at long and "unlimited-horizon" documents (SR-SWA sparse attention). That serving story is **SGLang**, not a 16 GB teaching GPU. Here you get the official single-page `infer()` call on short public pages — a receipt, a handwritten letter, and a results table — so you can see the Transformers API without standing up a server.

**Licence:** MIT.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. Hub shards are about **6.7 GB** (same size in float16). A single page whose longest edge is 1280 px, with `infer()` capped at up to 1024 new tokens, should leave headroom on 16 GB. That estimate is from the weight file plus a page, not a measured Colab run.

**Not in this notebook:** SGLang serving (the card's recommended production path), `infer_multi`, long multi-page or unlimited-horizon book demos, and the card's `torch==2.13.0` pin. Colab's installed torch is used as-is.

**Links:** [model card](https://huggingface.co/baidu/Unlimited-OCR-2) · [paper](https://arxiv.org/abs/2606.23050) · [GitHub](https://github.com/baidu/Unlimited-OCR) · [demo](https://huggingface.co/spaces/baidu/Unlimited-OCR-2)


In [ ]:
%pip install -q "transformers==4.52.1" "tokenizers==0.21.4" "huggingface-hub==0.36.0" "addict==2.4.0" "einops==0.8.2" "easydict==1.13" "pypdfium2==5.14.0" pillow

# The card tested Python 3.10, torch==2.13.0, transformers==4.52.1,
# huggingface-hub==0.36.0, tokenizers==0.21.4, addict, einops, and easydict
# on an A800-80GB. This notebook pins those model-side packages and leaves
# Colab's torch alone (do not force torch==2.13).
# Custom code needs trust_remote_code; the class is UnlimitedOCRForCausalLM
# via AutoModel (not AutoModelForCausalLM).
# pypdfium2 renders the PDF page. pillow is already on Colab.
# SGLang is not installed — it is a serving stack, not a free-T4 path.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 4.52.1 is invisible until the process restarts.


In [ ]:
import os

# Set before the first CUDA context when possible. After a restart this cell
# is the first torch import; the card uses expandable segments on long docs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
else:
    print("device: none (CPU). This notebook needs a T4 GPU.")

assert Version(transformers.__version__) == Version("4.52.1"), (
    f"transformers {transformers.__version__} is loaded; Unlimited-OCR-2's card "
    "pins 4.52.1. Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
# Forced float16 on T4. Do not follow the card's torch.bfloat16 A800 default.
DTYPE = torch.float16 if major < 8 else torch.bfloat16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
if major < 8:
    print("T4 path: float16 + official infer(). bfloat16 and SGLang are off.")


## 1. Load Unlimited-OCR-2

The card loads `AutoTokenizer` and `AutoModel` with `trust_remote_code=True`, `use_safetensors=True`, and `torch_dtype=torch.bfloat16`, then `.eval().cuda()`. On a T4 the only change is **float16**. The first run downloads about 6.7 GB.

`infer()` builds page tensors as `torch.bfloat16` in the shipped helper. After load we recast those tensors to the weight dtype so vision features match float16 weights. Single-page `infer()` uses **gundam** (dynamic tiles) by default; this notebook keeps that, with a T4 cap on tile count and new tokens.


In [ ]:
import inspect

from transformers import AutoModel, AutoTokenizer

MODEL_ID = "baidu/Unlimited-OCR-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=DTYPE,
).eval().cuda()

n_params = sum(p.numel() for p in model.parameters())
print("class:", type(model).__name__)
print(f"parameters: {n_params / 1e9:.2f}B | dtype: {next(model.parameters()).dtype}")

# Official infer() casts crops to bfloat16. Match the loaded weights on T4.
_uocr_mod = inspect.getmodule(type(model))
_orig_build = getattr(_uocr_mod, "_uocr_build_inputs", None)
if _orig_build is not None:
    def _build_inputs_matching_weights(*args, **kwargs):
        input_ids, imgs, seq_mask, spatial = _orig_build(*args, **kwargs)
        if imgs is not None:
            crops, globals_ = imgs
            imgs = (crops.to(dtype=DTYPE), globals_.to(dtype=DTYPE))
        return input_ids, imgs, seq_mask, spatial

    _uocr_mod._uocr_build_inputs = _build_inputs_matching_weights
    print("image tensors will be cast to", DTYPE, "(infer() builds them as bfloat16)")
else:
    print("no _uocr_build_inputs on the loaded module; infer() keeps its shipped dtypes")

print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")


## 2. Helpers and sample pages

Samples (same fixtures as the LightOnOCR / OvisOCR3 notebooks):

* the SROIE receipt from `hf-internal-testing/fixtures_ocr`
* a handwritten resignation letter ([CIPHR Connect](https://commons.wikimedia.org/wiki/File:Handwritten_Letter_of_Resignation.jpg), CC BY 2.0)
* page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762), which holds Table 2

`ocr_page` is a thin wrapper around the card's `model.infer(...)`. An empty prompt becomes `"<PAGE><image>Multi page parsing."` — that is the published single-page default. `eval_mode=True` turns off the token streamer so the cell prints the finished string. `max_new_tokens` is capped; the card default (`max_length - prompt_len`) is far too long for a T4 teaching cell.


In [ ]:
import io
import time

import requests
from IPython.display import display
from PIL import Image

RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
# Mirror of the Wikimedia Commons file (CC BY 2.0); upload.wikimedia.org often returns 429 to Colab IPs.
LETTER_URLS = [
    "https://raw.githubusercontent.com/donvito/notebooks/main/colab/data/lightonocr/handwritten-letter-of-resignation.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/e/eb/Handwritten_Letter_of_Resignation.jpg",
]
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
LONGEST = 1280  # T4 cap; the card does not require a resize
MAX_CROPS = 6  # official default is 16; fewer tiles on 16 GB
MAX_NEW = 1024


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"VRAM peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB{suffix}")
    else:
        print("device: none")


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def fetch_image(*urls):
    for url in urls:
        try:
            return Image.open(io.BytesIO(fetch_bytes(url))).convert("RGB")
        except requests.RequestException as error:
            print(f"Could not fetch {url}: {error}")
    raise RuntimeError("All image URLs failed")


def render_pdf_page(url, page_index, longest=LONGEST):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(fetch_bytes(url))
    image = pdf[page_index].render(scale=400 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def limit_edge(image, longest=LONGEST):
    image = image.copy()
    image.thumbnail((longest, longest))
    return image


def ocr_page(image, prompt="", max_new_tokens=MAX_NEW, crop_mode=True, max_crops=MAX_CROPS):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    text = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=image,
        crop_mode=crop_mode,
        max_crops=max_crops,
        max_new_tokens=max_new_tokens,
        eval_mode=True,
        save_results=False,
        temperature=0.0,
    )
    print_vram("after infer")
    return text


receipt = limit_edge(fetch_image(RECEIPT_URL))
letter = limit_edge(fetch_image(*LETTER_URLS))
table_page = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
print("receipt", receipt.size, "| letter", letter.size, "| table page", table_page.size)


### Receipt — single-page `infer()`

Same SROIE image as the LightOnOCR notebook. The call is the card's single-page `infer()` (gundam tiles, greedy, empty prompt). This is transcription, not LightOnOCR's `grounding` boxes or OvisOCR3's JSON schema.


In [ ]:
print("SROIE receipt from hf-internal-testing/fixtures_ocr")
display(receipt)
receipt_text = ocr_page(receipt, max_new_tokens=768)
print(receipt_text)


### Handwriting

One short letter. Still one `infer()` call. Do not send a multi-page stack here.


In [ ]:
print("Source: CIPHR Connect, CC BY 2.0, via Wikimedia Commons.")
display(letter)
letter_text = ocr_page(letter, max_new_tokens=900)
print(letter_text)


### Table — page 8 of Attention Is All You Need

One rendered PDF page. `infer_multi` and SGLang page-window scheduling are out of scope; this is still a single-page `infer()`.


In [ ]:
display(table_page)
table_text = ocr_page(table_page, max_new_tokens=1024)
print(table_text)


## 3. Caveats and cleanup

**What this notebook is for.** The official Transformers path on a free T4: `AutoModel` + `AutoTokenizer` + `infer()` on one page at a time. Use it to read a receipt, a letter, or a short table page and to see allocated / peak VRAM.

**What it is not.** SGLang is the card's serving recipe (custom wheel, page-window scheduling, n-gram logit processor). This notebook does not install it. `infer_multi` and long multi-page / unlimited-horizon runs are also skipped — they are the A800 / server story, not a 16 GB teaching GPU.

| | Unlimited-OCR-2 (this notebook) | LightOnOCR-3 0.8B / 4B | OvisOCR3 |
| --- | --- | --- | --- |
| Job | Single-page `infer()` transcription | Transcription + optional grounding boxes | Prompt-selected structured JSON |
| Runtime here | float16, official `infer()` | float16 + SDPA (4-bit on 4B) | float16 + SDPA |
| Long documents | Card: SGLang / `infer_multi` (not here) | One page per call | Two images max |

**T4 limits.** No native bfloat16, no FlashAttention 2, no SGLang. Weights stay in **float16**. Pages are limited to a 1280 px longest edge. `max_crops=6` (card default 16) and `max_new_tokens≤1024` (card default is the remaining context). If a generate OOMs, set `crop_mode=False` in `ocr_page` (one 1024 px global view, no tiles).

**dtype patch.** The shipped `_uocr_build_inputs` helper casts page tensors to `torch.bfloat16`. The load cell recasts them to the weight dtype. Without that, float16 weights and bfloat16 crops can disagree on a T4.

**Triton sparse kernels.** The checkpoint sets `dash_use_triton=true`. The entry module imports `sparse_gather_triton` / `sparse_gather_e2e_triton` only to copy those files into the Hugging Face cache; those imports are already `try/except`. DASH attention itself is documented as plain PyTorch. On a CPU-only import check, a Triton kernel can fail to compile and still be fine: this notebook's path is a **T4 GPU** with float16 `infer()`. If a fused kernel errors on the T4, the same `try/except` copy-hints apply — rerun the load cell and read the traceback; do not install a custom Triton wheel here.

**Card vs this notebook.** Official Transformers was tested on A800-80GB with `torch==2.13.0` and `torch.bfloat16`. Colab's torch is kept. Do not pin `torch==2.13` unless you are reproducing that exact A800 setup.

**Remote code.** `trust_remote_code=True` executes `modeling_unlimitedocr.py` and its DeepSeek / DASH helpers from the Hub repo. Review those files before you use this in a locked-down environment.

**Outputs can be wrong.** Small or blurred text, dense tables, and handwriting are the usual failure modes. `max_new_tokens` can truncate a busy page. Validate the string before you trust it.

**Samples.** The SROIE receipt and the Attention Is All You Need page are the same fixtures as the LightOnOCR / OvisOCR3 notebooks, so you can compare grounded markdown, schema JSON, and this model's raw `infer()` text. The letter is CIPHR Connect, CC BY 2.0.

**Cleanup.** *Runtime > Disconnect and delete runtime* drops the 6.7 GB checkpoint from the VM disk. Restarting the session keeps the files in the Hugging Face cache until the VM is recycled.
